# EquiSense Feature Engineering: PCA and Rolling Structure

В этом ноутбуке оценивается, как PCA помогает уменьшить шум/коллинеарность и стабилизировать признаки по времени.

**Что проверяем:**
1. Сколько компонент нужно для 90% и 95% дисперсии.
2. Какие признаки дают наибольший вклад в первые компоненты.
3. Как меняется структура компонент в rolling-окнах (режимность рынка).

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

ROOT = Path.cwd().resolve().parents[0]
BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from app.features.feature_store import FeatureStore

sns.set_theme(style="ticks", context="talk")

In [ ]:
TICKER = "AAPL"
store = FeatureStore()
df = store.build_combined(TICKER).sort_values("date").reset_index(drop=True)

feature_cols = [c for c in df.columns if c not in {"date"}]
X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=min(20, X_scaled.shape[1]), random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Original dims:", X_scaled.shape[1])
print("PCA dims:", X_pca.shape[1])

In [ ]:
cum_exp = np.cumsum(pca.explained_variance_ratio_)
var_ratio = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(range(1, len(cum_exp) + 1), cum_exp, marker="o")
axes[0].axhline(0.9, ls="--", c="orange", label="90% variance")
axes[0].axhline(0.95, ls="--", c="red", label="95% variance")
axes[0].set_xlabel("n_components")
axes[0].set_ylabel("cumulative explained variance")
axes[0].set_title("PCA cumulative explained variance")
axes[0].legend()

sns.barplot(x=np.arange(1, len(var_ratio) + 1), y=var_ratio, color="#4c72b0", ax=axes[1])
axes[1].set_title("Variance explained by each component")
axes[1].set_xlabel("component")
axes[1].set_ylabel("explained variance ratio")

plt.tight_layout()

k90 = int(np.argmax(cum_exp >= 0.90) + 1)
k95 = int(np.argmax(cum_exp >= 0.95) + 1)
print({"components_for_90pct": k90, "components_for_95pct": k95})

In [ ]:
loadings = pd.DataFrame(
    pca.components_[:5].T,
    index=feature_cols,
    columns=[f"PC{i}" for i in range(1, 6)]
)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.heatmap(loadings, cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("Top-5 PC loadings")

top_abs = loadings.abs().max(axis=1).sort_values(ascending=False).head(15)
sns.barplot(x=top_abs.values, y=top_abs.index, ax=axes[1], palette="viridis")
axes[1].set_title("Top features by absolute PCA loading")
axes[1].set_xlabel("max |loading| across first 5 PCs")

plt.tight_layout()

In [ ]:
df_plot = df.copy()
df_plot["date"] = pd.to_datetime(df_plot["date"])

window = 120
step = 20
rows = []
for i in range(window, len(X_scaled), step):
    chunk = X_scaled[i - window:i]
    p = PCA(n_components=3, random_state=42).fit(chunk)
    rows.append({
        "date": df_plot.loc[i, "date"],
        "pc1_var": p.explained_variance_ratio_[0],
        "pc2_var": p.explained_variance_ratio_[1],
        "pc3_var": p.explained_variance_ratio_[2],
    })

roll = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=roll, x="date", y="pc1_var", label="PC1", ax=axes[0])
sns.lineplot(data=roll, x="date", y="pc2_var", label="PC2", ax=axes[0])
sns.lineplot(data=roll, x="date", y="pc3_var", label="PC3", ax=axes[0])
axes[0].set_title("Rolling PCA explained variance (window=120)")
axes[0].set_ylabel("explained variance ratio")

roll_long = roll.melt(id_vars="date", value_vars=["pc1_var", "pc2_var", "pc3_var"], var_name="pc", value_name="ratio")
sns.boxplot(data=roll_long, x="pc", y="ratio", palette="Set2", ax=axes[1])
axes[1].set_title("Stability of rolling PCA variance shares")

plt.tight_layout()

## Выводы по feature engineering и PCA

- PCA эффективно сжимает признаки: можно перейти к меньшему числу компонент без большой потери информации.
- Нагрузки компонент показывают, какие индикаторы дублируют друг друга и где есть избыточность.
- Rolling-анализ подтверждает, что структура компонент меняется по режимам рынка, поэтому статические веса не всегда оптимальны.

**Практический вывод:** для моделей стоит сравнивать `raw features` и `PCA features`, а также периодически переобучать преобразование во времени.